# Baseline Sentiment Model

This notebook builds a classical NLP baseline for three-class sentiment classification using Financial PhraseBank.

In [ ]:
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

## Data preparation

In [ ]:
dataset = load_dataset(
    "takala/financial_phrasebank",
    "sentences_75agree",
    trust_remote_code=True,
)

df = (
    dataset["train"]
    .to_pandas()
    .drop_duplicates(subset="sentence")
    .reset_index(drop=True)
)

X = df["sentence"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

pd.Series({"train_rows": len(X_train), "test_rows": len(X_test)})

## TF-IDF feature extraction

In [ ]:
def tokenize(sentence):
    return re.findall(r"\b\w+\b", sentence.lower())


def calculate_tf(word_count, document_length):
    if document_length == 0:
        return 0.0
    return word_count / document_length


def calculate_idf(number_of_documents, document_frequency):
    return math.log(
        (1 + number_of_documents) / (1 + document_frequency)
    ) + 1


def calculate_tfidf(tf, idf):
    return tf * idf


def fit_tfidf(sentences, minimum_document_frequency=2):
    tokenized_documents = [tokenize(sentence) for sentence in sentences]

    document_frequencies = Counter()
    for tokens in tokenized_documents:
        document_frequencies.update(set(tokens))

    vocabulary = sorted(
        word
        for word, frequency in document_frequencies.items()
        if frequency >= minimum_document_frequency
    )
    word_to_index = {
        word: index for index, word in enumerate(vocabulary)
    }

    number_of_documents = len(sentences)
    idf_values = {
        word: calculate_idf(
            number_of_documents, document_frequencies[word]
        )
        for word in vocabulary
    }

    return word_to_index, idf_values


def transform_to_tfidf(sentences, word_to_index, idf_values):
    row_indices = []
    column_indices = []
    values = []

    for row_index, sentence in enumerate(sentences):
        tokens = tokenize(sentence)
        word_counts = Counter(tokens)
        document_length = len(tokens)

        for word, word_count in word_counts.items():
            column_index = word_to_index.get(word)
            if column_index is None:
                continue

            tf = calculate_tf(word_count, document_length)
            value = calculate_tfidf(tf, idf_values[word])

            row_indices.append(row_index)
            column_indices.append(column_index)
            values.append(value)

    return csr_matrix(
        (values, (row_indices, column_indices)),
        shape=(len(sentences), len(word_to_index)),
        dtype=np.float64,
    )

In [ ]:
word_to_index, idf_values = fit_tfidf(X_train)
vocabulary = list(word_to_index)

X_train_tfidf = transform_to_tfidf(
    X_train, word_to_index, idf_values
)
X_test_tfidf = transform_to_tfidf(
    X_test, word_to_index, idf_values
)

In [ ]:
# Equivalent implementation using scikit-learn's built-in vectorizer:
#
# from sklearn.feature_extraction.text import TfidfVectorizer
#
# tfidf_vectorizer = TfidfVectorizer(
#     lowercase=True,
#     ngram_range=(1, 2),
#     min_df=2,
#     max_df=0.95,
#     sublinear_tf=True,
# )
#
# X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
# X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [ ]:
feature_summary = pd.Series(
    {
        "training_documents": X_train_tfidf.shape[0],
        "test_documents": X_test_tfidf.shape[0],
        "tfidf_features": X_train_tfidf.shape[1],
        "matrix_density": X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]),
    }
)

feature_summary

The vocabulary and IDF values are learned exclusively from the training set to prevent information from leaking from the held-out test set.

## Logistic Regression baseline

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

baseline_model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = baseline_model.predict(X_test_tfidf)

prediction_preview = pd.DataFrame(
    {
        "sentence": X_test.iloc[:10].values,
        "actual_label": y_test.iloc[:10].values,
        "predicted_label": y_pred[:10],
    }
)

prediction_preview

## Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

evaluation_summary = pd.Series(
    {
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
    }
).round(3)

evaluation_summary

In [ ]:
label_names = ["negative", "neutral", "positive"]

class_report = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2],
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )
).transpose().round(3)

class_report

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=[0, 1, 2],
    display_labels=label_names,
    cmap="Blues",
)
plt.title("Logistic Regression confusion matrix")
plt.tight_layout()
plt.show()

## Class-balanced Logistic Regression

In [ ]:
balanced_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)

balanced_model.fit(X_train_tfidf, y_train)
balanced_pred = balanced_model.predict(X_test_tfidf)

In [ ]:
from sklearn.metrics import recall_score

def summarize_model(y_true, predictions):
    class_recalls = recall_score(
        y_true, predictions, labels=[0, 1, 2], average=None
    )
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "macro_f1": f1_score(y_true, predictions, average="macro"),
        "negative_recall": class_recalls[0],
        "neutral_recall": class_recalls[1],
        "positive_recall": class_recalls[2],
    }

model_comparison = pd.DataFrame(
    {
        "baseline": summarize_model(y_test, y_pred),
        "class_balanced": summarize_model(y_test, balanced_pred),
    }
).transpose().round(3)

model_comparison

In [ ]:
balanced_report = pd.DataFrame(
    classification_report(
        y_test,
        balanced_pred,
        labels=[0, 1, 2],
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )
).transpose().round(3)

balanced_report

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    balanced_pred,
    labels=[0, 1, 2],
    display_labels=label_names,
    cmap="Oranges",
)
plt.title("Class-balanced Logistic Regression confusion matrix")
plt.tight_layout()
plt.show()

## Model comparison conclusion

Class weighting increased negative recall from 0.464 to 0.702 and macro F1 from 0.743 to 0.781, while accuracy decreased only slightly from 0.828 to 0.826. The class-balanced model provides substantially more even performance across sentiments and is the preferred candidate for further analysis.

## Stratified cross-validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

cross_validator = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)
X_cv = X_train.reset_index(drop=True)
y_cv = y_train.reset_index(drop=True)
cv_records = []

for fold, (fold_train_indices, fold_validation_indices) in enumerate(
    cross_validator.split(X_cv, y_cv), start=1
):
    X_fold_train = X_cv.iloc[fold_train_indices]
    X_fold_validation = X_cv.iloc[fold_validation_indices]
    y_fold_train = y_cv.iloc[fold_train_indices]
    y_fold_validation = y_cv.iloc[fold_validation_indices]

    fold_word_to_index, fold_idf_values = fit_tfidf(X_fold_train)
    X_fold_train_tfidf = transform_to_tfidf(
        X_fold_train, fold_word_to_index, fold_idf_values
    )
    X_fold_validation_tfidf = transform_to_tfidf(
        X_fold_validation, fold_word_to_index, fold_idf_values
    )

    configurations = {
        "baseline": None,
        "class_balanced": "balanced",
    }
    for model_name, class_weight in configurations.items():
        fold_model = LogisticRegression(
            class_weight=class_weight,
            max_iter=1000,
            random_state=42,
        )
        fold_model.fit(X_fold_train_tfidf, y_fold_train)
        fold_predictions = fold_model.predict(X_fold_validation_tfidf)
        cv_records.append(
            {
                "fold": fold,
                "model": model_name,
                "accuracy": accuracy_score(
                    y_fold_validation, fold_predictions
                ),
                "macro_f1": f1_score(
                    y_fold_validation, fold_predictions, average="macro"
                ),
            }
        )

cv_results = pd.DataFrame(cv_records)
cv_results.round(3)

In [ ]:
cv_summary = (
    cv_results.groupby("model")[["accuracy", "macro_f1"]]
    .agg(["mean", "std"])
    .round(3)
)

cv_summary

The custom TF-IDF vocabulary and IDF values are rebuilt inside every fold. Model selection therefore uses validation data that did not influence feature extraction or fitting.

The class-balanced model achieved a higher mean macro F1 across all five folds (0.762 versus 0.714) and slightly higher mean accuracy (0.815 versus 0.810). Cross-validation therefore supports selecting class-balanced Logistic Regression.

## Error analysis

In [ ]:
label_mapping = {0: "negative", 1: "neutral", 2: "positive"}

prediction_errors = pd.DataFrame(
    {
        "sentence": X_test.to_numpy(),
        "actual": y_test.map(label_mapping).to_numpy(),
        "predicted": [label_mapping[label] for label in balanced_pred],
    }
)
prediction_errors = prediction_errors.query("actual != predicted")

error_counts = pd.crosstab(
    prediction_errors["actual"],
    prediction_errors["predicted"],
    rownames=["actual"],
    colnames=["predicted"],
)
error_counts

In [ ]:
prediction_errors.sample(
    n=min(15, len(prediction_errors)),
    random_state=42,
).reset_index(drop=True)

## Feature interpretation

In [ ]:
feature_names = np.asarray(vocabulary)
top_feature_rows = []

for class_index, class_label in enumerate(balanced_model.classes_):
    top_indices = np.argsort(balanced_model.coef_[class_index])[-15:][::-1]
    for rank, feature_index in enumerate(top_indices, start=1):
        top_feature_rows.append(
            {
                "sentiment": label_mapping[int(class_label)],
                "rank": rank,
                "term": feature_names[feature_index],
                "coefficient": balanced_model.coef_[
                    class_index, feature_index
                ],
            }
        )

top_features = pd.DataFrame(top_feature_rows)
top_features.pivot(index="rank", columns="sentiment", values="term")

## Phase 1 conclusion

The custom TF-IDF and class-balanced Logistic Regression pipeline provides a reproducible classical NLP baseline. Cross-validation supports class weighting, and the held-out results show materially improved minority-class recall with minimal loss in accuracy. Remaining limitations primarily concern dataset size, sentence-level context, and the representational limits of unigram features.